## Stage 1 Training

In [1]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade_from_scratch \
#   --quality-loss \
#   --use-highres-refine \
#   --epochs 24 \
#   --batch-size 8 \
#   --lr 5e-5 \
#   --diagnostics-after-training \
#   --diag-save-dir runs/stage1_quality_upgrade_from_scratch/diagnostics_stage1 \
#   --diag-balanced-samples-per-class 8 \
#   --diag-max-batches 50 \
#   --diag-num-samples 12

In [2]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade \
#   --warm-start runs/stage1_quality_upgrade_from_scratch/checkpoints/stage1_best.pt \
#   --allow-partial-load \
#   --quality-loss \
#   --use-highres-refine \
#   --epochs 24 \
#   --batch-size 8 \
#   --lr 5e-5 \
#   --presence-threshold 0.25 \
#   --topk-presence-k 128 \
#   --small-part-area-tau 0.015 \
#   --small-part-weight-max 6.0 \
#   --small-part-weight-power 0.5 \
#   --quality-presence-bce 0.40 \
#   --valid-absent-topmean-fp 0.08 \
#   --valid-absent-mean-fp 0.02 \
#   --invalid-part-topmean 0.35 \
#   --invalid-part-mean 0.08 \
#   --gt-support-leak 0.35 \
#   --pred-support-containment 0.25 \
#   --boundary-loss 0.08 \
#   --focal-functional 0.12 \
#   --tversky-functional 0.12 \
#   --quality-topq 0.02 \
#   --diagnostics-after-training \
#   --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1 \
#   --diag-balanced-samples-per-class 8 \
#   --diag-max-batches 50 \
#   --diag-num-samples 12 \
#   --diag-max-parts-per-sample 8 \
#   --diag-mask-threshold 0.40

## 4. Diagnostics-only run for an existing checkpoint

In [3]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade \
#   --diag-checkpoint runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
#   --allow-partial-load \
#   --quality-loss \
#   --use-highres-refine \
#   --presence-threshold 0.25 \
#   --topk-presence-k 128 \
#   --diagnostics-only \
#   --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1 \
#   --diag-balanced-samples-per-class 8 \
#   --diag-max-batches 50 \
#   --diag-num-samples 12 \
#   --diag-max-parts-per-sample 8 \
#   --diag-mask-threshold 0.40 \
#   --diag-presence-thresholds 0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50

## 5. Fast diagnostics on a small subset

In [4]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade \
#   --diag-checkpoint runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
#   --allow-partial-load \
#   --quality-loss \
#   --use-highres-refine \
#   --presence-threshold 0.25 \
#   --topk-presence-k 128 \
#   --diagnostics-only \
#   --diag-save-dir runs/stage1_quality_upgrade/diagnostics_stage1_fast \
#   --diag-balanced-samples-per-class 2 \
#   --diag-max-batches 8 \
#   --diag-num-samples 6 \
#   --diag-max-parts-per-sample 8 \
#   --diag-mask-threshold 0.40

## Build HKG

In [5]:
!PYTHONPATH=src python scripts/cache_strict_aog_terminals.py \
  --config configs/stage1_quality_upgrade.yaml \
  --stage1-ckpt runs/stage1_quality_upgrade/checkpoints/stage1_best.pt \
  --out-dir runs/strict_aog_cache \
  --device auto \
  --splits train,val \
  --batch-size 16 \
  --threshold 0.40 \
  --max-components-per-part 4 \
  --max-terminals 32

[dataset] train.json: 20457 samples | C=11 F=13 R=40
[dataset] val.json: 1205 samples | C=11 F=13 R=40
[datasets] train: ../full_hyco/PartImageNet/annotations/train/train.json | images: ../full_hyco/PartImageNet/images/train | samples: 20457
[datasets] val: ../full_hyco/PartImageNet/annotations/val/val.json | images: ../full_hyco/PartImageNet/images/val | samples: 1205
/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/dfli/anaconda3/envs/partviz/lib/python3.12/

In [6]:
!PYTHONPATH=src python scripts/build_strict_aog.py \
  --config configs/stage1_quality_upgrade.yaml \
  --cache runs/strict_aog_cache/train_strict_aog_terminals.pt \
  --out runs/strict_aog_cache/strict_aog.pt \
  --num-templates-per-class 3 \
  --max-slots-per-template 12 \
  --max-slots-per-part 4

saved strict AOG to runs/strict_aog_cache/strict_aog.pt
classes=11 templates=3 slots=12 edges=287
valid_templates=33 valid_slots=241


## Train Stage 2

In [7]:
!PYTHONPATH=src python scripts/train_strict_aog.py \
  --grammar runs/strict_aog_cache/strict_aog.pt \
  --train-cache runs/strict_aog_cache/train_strict_aog_terminals.pt \
  --val-cache runs/strict_aog_cache/val_strict_aog_terminals.pt \
  --save-dir runs/strict_aog \
  --device auto \
  --batch-size 64 \
  --assignment sinkhorn

[strict-aog] epoch=1 train_loss=0.1533 train_acc=0.9639 val_acc=0.9602 logit_std=4.169434
[strict-aog] epoch=2 train_loss=0.1089 train_acc=0.9736 val_acc=0.9552 logit_std=4.565374
[strict-aog] epoch=3 train_loss=0.0965 train_acc=0.9747 val_acc=0.9568 logit_std=4.757687
[strict-aog] epoch=4 train_loss=0.0883 train_acc=0.9762 val_acc=0.9618 logit_std=5.058891
[strict-aog] epoch=5 train_loss=0.0835 train_acc=0.9777 val_acc=0.9627 logit_std=5.257218
[strict-aog] epoch=6 train_loss=0.0787 train_acc=0.9810 val_acc=0.9635 logit_std=5.403142
[strict-aog] epoch=7 train_loss=0.0759 train_acc=0.9817 val_acc=0.9618 logit_std=5.495198
[strict-aog] epoch=8 train_loss=0.0727 train_acc=0.9825 val_acc=0.9618 logit_std=5.636669
[strict-aog] epoch=9 train_loss=0.0707 train_acc=0.9826 val_acc=0.9627 logit_std=5.692409
[strict-aog] epoch=10 train_loss=0.0690 train_acc=0.9835 val_acc=0.9643 logit_std=5.868644
[strict-aog] epoch=11 train_loss=0.0671 train_acc=0.9846 val_acc=0.9627 logit_std=5.938078
[strict-

In [8]:
!PYTHONPATH=src python scripts/train_strict_aog.py \
  --grammar runs/strict_aog_cache/strict_aog.pt \
  --train-cache runs/strict_aog_cache/train_strict_aog_terminals.pt \
  --val-cache runs/strict_aog_cache/val_strict_aog_terminals.pt \
  --save-dir runs/strict_aog_max \
  --device auto \
  --batch-size 128 \
  --assignment max

[strict-aog] epoch=1 train_loss=0.1613 train_acc=0.9664 val_acc=0.9502 logit_std=5.502125
[strict-aog] epoch=2 train_loss=0.1300 train_acc=0.9742 val_acc=0.9494 logit_std=5.256697
[strict-aog] epoch=3 train_loss=0.1228 train_acc=0.9758 val_acc=0.9527 logit_std=5.093021
[strict-aog] epoch=4 train_loss=0.1189 train_acc=0.9774 val_acc=0.9535 logit_std=4.969239
[strict-aog] epoch=5 train_loss=0.1152 train_acc=0.9789 val_acc=0.9535 logit_std=4.883064
[strict-aog] epoch=6 train_loss=0.1129 train_acc=0.9789 val_acc=0.9552 logit_std=4.792653
[strict-aog] epoch=7 train_loss=0.1111 train_acc=0.9795 val_acc=0.9519 logit_std=4.692311
[strict-aog] epoch=8 train_loss=0.1095 train_acc=0.9807 val_acc=0.9535 logit_std=4.675311
[strict-aog] epoch=9 train_loss=0.1082 train_acc=0.9809 val_acc=0.9519 logit_std=4.626889
[strict-aog] epoch=10 train_loss=0.1078 train_acc=0.9803 val_acc=0.9544 logit_std=4.613327
[strict-aog] epoch=11 train_loss=0.1066 train_acc=0.9811 val_acc=0.9544 logit_std=4.549284
[strict-